# Paso 3 — Cargar las Dimensiones (Silver → Gold)
**Archivo original:** `etl/ETL-Dimension.py`

---

## ¿Qué hace este script?

Lee la tabla de staging (`stg_aduana`) y genera las **12 tablas de dimensiones**.

**Patrón común para cada dimensión:**
1. `DELETE` — limpiar la dimensión
2. `INSERT INTO dim_xxx ... SELECT DISTINCT ... FROM stg_aduana` — extraer valores únicos
3. `ROW_NUMBER() OVER (ORDER BY ...)` — asignar ID surrogate secuencial

**¿Por qué `SELECT DISTINCT`?**
Porque en el staging hay miles de filas, pero solo queremos los valores únicos
para cada dimensión. Por ejemplo, si hay 250 despachos pero solo 3 aduanas,
la `dim_aduana` tendrá solo 3 filas.


In [ ]:
import duckdb

DB_PATH = r"C:\Información\proyectos\aduana_bi\db\aduana.duckdb"
con = duckdb.connect(DB_PATH)
print("Conectado. Generando dimensiones...")

---

## dim_operacion

Solo tiene 2 valores posibles: `IMPORTACION` y `EXPORTACION`.

Se agregan dos columnas booleanas derivadas (`es_importacion`, `es_exportacion`)
para facilitar los filtros en Power BI sin necesidad de recordar el texto exacto.

**Técnica SQL usada:**
`UPPER(TRIM(operacion)) = 'IMPORTACION'` devuelve `TRUE` o `FALSE`.
El `TRIM()` elimina espacios al inicio/fin que pueden venir del Excel.


In [ ]:
con.execute("DELETE FROM dw.dim_operacion;")
con.execute("""
INSERT INTO dw.dim_operacion (id_operacion, operacion, es_importacion, es_exportacion)
SELECT
    ROW_NUMBER() OVER (ORDER BY operacion) AS id_operacion,  -- ID: 1, 2, 3...
    operacion,
    UPPER(TRIM(operacion)) = 'IMPORTACION' AS es_importacion,
    UPPER(TRIM(operacion)) = 'EXPORTACION' AS es_exportacion
FROM (
    SELECT DISTINCT operacion          -- un valor único por fila
    FROM dw.stg_aduana
    WHERE operacion IS NOT NULL
      AND TRIM(operacion) <> ''        -- filtrar vacíos
) t;
""")

print(con.execute("SELECT * FROM dw.dim_operacion").fetchdf().to_string())

---

## dim_destinacion

Esta dimensión se **enriquece** cruzando dos fuentes:
- `stg_aduana.destinacion` → tiene los códigos usados en los despachos
- `stg_destinaciones` → tiene la descripción y tipo de cada código

Se usa un `LEFT JOIN` porque puede haber códigos en los despachos
que no estén en el catálogo (quedarán con descripción NULL).


In [ ]:
con.execute("DELETE FROM dw.dim_destinacion;")
con.execute("""
INSERT INTO dw.dim_destinacion (
    id_destinacion, cod_destinacion, descripcion_dest, tipo_regimen_base, tipo_operacion_base
)
SELECT
    ROW_NUMBER() OVER (ORDER BY s.destinacion) AS id_destinacion,
    s.destinacion AS cod_destinacion,
    d.descripcion_dest,          -- puede ser NULL si no está en el catálogo
    d.tipo_regimen_base,
    d.tipo_operacion_base
FROM (
    SELECT DISTINCT destinacion
    FROM dw.stg_aduana
    WHERE destinacion IS NOT NULL
      AND TRIM(destinacion) <> ''
) s
LEFT JOIN dw.stg_destinaciones d
    ON TRIM(s.destinacion) = TRIM(d.cod_destinacion);  -- TRIM para evitar espacios
""")

n = con.execute("SELECT COUNT(*) FROM dw.dim_destinacion").fetchone()[0]
print(f"dim_destinacion: {n} filas")

---

## dim_regimen, dim_aduana — Dimensiones simples

Estas dimensiones son las más simples: solo un campo único de texto.
El patrón es idéntico: `SELECT DISTINCT campo FROM stg_aduana`.


In [ ]:
for dim, campo in [("dim_regimen", "regimen"), ("dim_aduana", "aduana")]:
    con.execute(f"DELETE FROM dw.{dim};")
    con.execute(f"""
    INSERT INTO dw.{dim} (id_{dim.replace('dim_','')}, {campo})
    SELECT
        ROW_NUMBER() OVER (ORDER BY {campo}),
        {campo}
    FROM (
        SELECT DISTINCT {campo}
        FROM dw.stg_aduana
        WHERE {campo} IS NOT NULL AND TRIM({campo}) <> ''
    ) t;
    """)
    n = con.execute(f"SELECT COUNT(*) FROM dw.{dim}").fetchone()[0]
    print(f"{dim}: {n} filas")

---

## dim_pais — Técnica de parseo de cadena

En el staging, los países vienen en un formato combinado: `'ARG - ARGENTINA'`
(código + descripción en el mismo campo).

**Técnica SQL:** `SPLIT_PART(campo, ' - ', posicion)`
- `SPLIT_PART('ARG - ARGENTINA', ' - ', 1)` → `'ARG'` (código)
- `SPLIT_PART('ARG - ARGENTINA', ' - ', 2)` → `'ARGENTINA'` (descripción)

También se usa `UNION` (sin ALL) para combinar países origen y destino
en una única dimensión sin duplicados.


In [ ]:
con.execute("DELETE FROM dw.dim_pais;")
con.execute("""
INSERT INTO dw.dim_pais (id_pais, codigo_pais, descripcion_pais)
SELECT
    ROW_NUMBER() OVER (ORDER BY codigo_pais, descripcion_pais) AS id_pais,
    codigo_pais,
    descripcion_pais
FROM (
    SELECT DISTINCT
        TRIM(SPLIT_PART(pais, ' - ', 1)) AS codigo_pais,      -- extrae 'ARG'
        TRIM(SPLIT_PART(pais, ' - ', 2)) AS descripcion_pais  -- extrae 'ARGENTINA'
    FROM (
        -- UNION (sin ALL) elimina duplicados entre origen y destino
        SELECT pais_origen AS pais FROM dw.stg_aduana
        UNION
        SELECT pais_procedencia_destino AS pais FROM dw.stg_aduana
    ) x
    WHERE pais IS NOT NULL
      AND TRIM(pais) <> ''
) t
WHERE codigo_pais IS NOT NULL
  AND TRIM(codigo_pais) <> '';
""")

n = con.execute("SELECT COUNT(*) FROM dw.dim_pais").fetchone()[0]
print(f"dim_pais: {n} países")
print(con.execute("SELECT * FROM dw.dim_pais LIMIT 5").fetchdf().to_string())

---

## dim_producto — Clave compuesta

Un producto no se identifica solo por el código NCM (`posicion`),
sino por la **combinación de 6 campos**: posicion, rubro, desc_capitulo,
desc_posicion, desc_partida y mercaderia.

**¿Por qué?** Porque el mismo código NCM puede tener distintas
descripciones de mercadería según el despacho.

Se usa `COALESCE(campo, '')` para tratar los NULL como cadena vacía
al ordenar, evitando que los NULLs alteren el orden del `ROW_NUMBER()`.


In [ ]:
con.execute("DELETE FROM dw.dim_producto;")
con.execute("""
INSERT INTO dw.dim_producto (
    id_producto, posicion_ncm, rubro, desc_capitulo, desc_posicion, desc_partida, mercaderia
)
SELECT
    ROW_NUMBER() OVER (
        ORDER BY
            COALESCE(posicion, ''),
            COALESCE(rubro, ''),
            COALESCE(desc_capitulo, ''),
            COALESCE(desc_posicion, ''),
            COALESCE(desc_partida, ''),
            COALESCE(mercaderia, '')
    ) AS id_producto,
    posicion AS posicion_ncm,
    rubro,
    desc_capitulo,
    desc_posicion,
    desc_partida,
    mercaderia
FROM (
    SELECT DISTINCT posicion, rubro, desc_capitulo, desc_posicion, desc_partida, mercaderia
    FROM dw.stg_aduana
) t;
""")

n = con.execute("SELECT COUNT(*) FROM dw.dim_producto").fetchone()[0]
print(f"dim_producto: {n} productos únicos")

---

## Resto de dimensiones simples


In [ ]:
dimensiones_simples = [
    ("dim_medio_transporte", "id_medio_transporte", "medio_transporte",          "medio_transporte"),
    ("dim_canal",            "id_canal",            "canal",                     "canal"),
    ("dim_unidad_medida",    "id_unidad_medida",    "unidad_medida",             "unidad_medida_estadistica"),
    ("dim_acuerdo",          "id_acuerdo",          "acuerdo",                   "acuerdo"),
    ("dim_marca",            "id_marca",            "marca",                     "marca_item"),
]

for dim, id_col, dest_col, src_col in dimensiones_simples:
    con.execute(f"DELETE FROM dw.{dim};")
    con.execute(f"""
    INSERT INTO dw.{dim} ({id_col}, {dest_col})
    SELECT
        ROW_NUMBER() OVER (ORDER BY {src_col}),
        {src_col} AS {dest_col}
    FROM (
        SELECT DISTINCT {src_col}
        FROM dw.stg_aduana
        WHERE {src_col} IS NOT NULL AND TRIM({src_col}) <> ''
    ) t;
    """)
    n = con.execute(f"SELECT COUNT(*) FROM dw.{dim}").fetchone()[0]
    print(f"{dim}: {n} filas")

---

## dim_fecha — Dimensión de tiempo

La dimensión de fecha es especial: se construye a partir de las fechas
reales que aparecen en los datos (`oficializacion` y `cancelacion`).

Para cada fecha única se calculan atributos derivados:
- `EXTRACT(YEAR FROM f)` → año (ej: 2024)
- `EXTRACT(MONTH FROM f)` → mes número (ej: 3)
- `STRFTIME(f, '%B')` → nombre del mes (ej: 'March')
- `FLOOR((mes - 1) / 3) + 1` → trimestre (ej: 1)
- `STRFTIME(f, '%Y-%m')` → año-mes para ordenar (ej: '2024-03')

Se usa una **CTE** (`WITH fechas AS (...)`) para combinar ambas fechas
antes de calcular los atributos.


In [ ]:
con.execute("DELETE FROM dw.dim_fecha;")
con.execute("""
INSERT INTO dw.dim_fecha (
    id_fecha, fecha, anio, mes_numero, mes_nombre, trimestre, anio_mes
)
WITH fechas AS (
    -- Combinar todas las fechas de oficialización y cancelación
    SELECT oficializacion AS f FROM dw.stg_aduana WHERE oficializacion IS NOT NULL
    UNION
    SELECT cancelacion    FROM dw.stg_aduana WHERE cancelacion IS NOT NULL
)
SELECT
    ROW_NUMBER() OVER (ORDER BY f) AS id_fecha,
    f AS fecha,
    EXTRACT(YEAR FROM f),
    EXTRACT(MONTH FROM f),
    STRFTIME(f, '%B'),                                             -- nombre mes en inglés
    CAST(FLOOR((EXTRACT(MONTH FROM f) - 1) / 3) + 1 AS INTEGER), -- trimestre: 1,2,3,4
    STRFTIME(f, '%Y-%m')                                          -- '2024-03'
FROM fechas;
""")

n = con.execute("SELECT COUNT(*) FROM dw.dim_fecha").fetchone()[0]
print(f"dim_fecha: {n} fechas únicas")
print(con.execute("SELECT * FROM dw.dim_fecha LIMIT 5").fetchdf().to_string())

---

## Resumen de dimensiones generadas


In [ ]:
con.close()
print("Dimensiones generadas correctamente.")

In [ ]:
con = duckdb.connect(DB_PATH)
dims = [
    "dim_operacion", "dim_destinacion", "dim_regimen", "dim_aduana",
    "dim_pais", "dim_producto", "dim_medio_transporte", "dim_canal",
    "dim_unidad_medida", "dim_acuerdo", "dim_marca", "dim_fecha"
]
print(f"{'Dimensión':<30} {'Filas':>8}")
print("-" * 40)
for d in dims:
    n = con.execute(f"SELECT COUNT(*) FROM dw.{d}").fetchone()[0]
    print(f"{d:<30} {n:>8}")
con.close()

---

**Siguiente paso:** `04_Cargar_Fact_Table.ipynb`
